# Lesson 9b: Attention — Practical

9a derived attention from the alignment problem a fixed-size encoder
state creates, and verified the mechanism itself against PyTorch. This
notebook trains two real sequence-to-sequence models on the same
synthetic task — one a vanilla encoder-decoder using only the encoder's
final hidden state (9a's bottleneck, concretely), the other identical
except the decoder attends over every encoder state at every step — and
measures the gap attention actually buys. Both are trained with
`nn.LSTM`, teacher forcing and Adam, exactly like 7b.

By the end of this notebook you will have:
- trained a **vanilla seq2seq baseline and an attention-augmented seq2seq
  model** on the same synthetic task and **measured attention's accuracy
  advantage** directly,
- **plotted and interpreted an attention heatmap**, checking it recovers
  the task's true alignment without ever being told what that alignment
  is, and
- shown how **masking** changes the attention distribution when part of
  the input is padding rather than real content.

## Introduction

The task below — sequence reversal — is deliberately chosen to make
9a's alignment argument concrete rather than abstract: decoder output
position $t$ depends on a *different, specific* encoder position ($L-1-t$)
for every single step. A vanilla decoder has to reconstruct all $L$ of
those correspondences from one fixed final hidden state; an attention
decoder can instead look directly at the one encoder position it
actually needs, at the moment it needs it.

## Setup

In [ ]:
# Fixed seeds: every stochastic step (weight init, synthetic data
# sampling) is reproducible.
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 4)
device = torch.device("cpu")
print("numpy:", np.__version__)
print("torch:", torch.__version__)

## A Seq2Seq Task

**Sequence reversal**: given a random sequence of $L$ digits, produce
the exact same digits in reverse order. Trivial for a human, and
deliberately hard for a fixed-size bottleneck — the correct output at
every position is a direct copy of one specific input position, so a
model with direct access to every encoder state has an enormous
structural advantage over one that must route everything through a
single vector first. Data is generated fresh on every batch (an
essentially inexhaustible synthetic task), so there is no train/test
split to manage — every batch is effectively held out.

In [ ]:
VOCAB_SIZE = 12  # digits 0-9, BOS=10, PAD=11 (PAD only used in the masking demo)
BOS, PAD = 10, 11
L = 16  # sequence length -- long enough that a single fixed vector is a real bottleneck


def make_batch(batch_size, seq_len, rng):
    sources = rng.integers(0, 10, size=(batch_size, seq_len))
    targets = sources[:, ::-1].copy()
    decoder_in = np.concatenate([np.full((batch_size, 1), BOS), targets[:, :-1]], axis=1)
    return (torch.tensor(sources, dtype=torch.long),
            torch.tensor(decoder_in, dtype=torch.long),
            torch.tensor(targets, dtype=torch.long))


rng = np.random.default_rng(SEED)
src, dec_in, tgt = make_batch(2, L, rng)
print("source:     ", src[0].tolist())
print("decoder in: ", dec_in[0].tolist())
print("target:     ", tgt[0].tolist())

### A baseline with no attention

The encoder's *entire* summary of the source sequence is its final
hidden and cell state — exactly the fixed-size bottleneck 9a measured
the sensitivity of directly.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        outputs, (h, c) = self.lstm(self.embed(x))
        return outputs, h.squeeze(0), c.squeeze(0)  # outputs: (B, L, H); h, c: (B, H)


class BaselineDecoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.cell = nn.LSTMCell(emb_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, decoder_in, h, c):
        logits = []
        for t in range(decoder_in.shape[1]):
            h, c = self.cell(self.embed(decoder_in[:, t]), (h, c))
            logits.append(self.out(h))
        return torch.stack(logits, dim=1)  # (B, L, vocab_size)


EMB_DIM, HIDDEN_DIM = 16, 64

def token_accuracy(logits, targets):
    return (logits.argmax(dim=-1) == targets).float().mean().item()


def train_model(make_forward_fn, n_steps=3000, batch_size=64, lr=1e-3, seed=SEED):
    train_rng = np.random.default_rng(seed)
    modules, forward_fn = make_forward_fn()
    params = [p for m in modules for p in m.parameters()]
    optimizer = torch.optim.Adam(params, lr=lr)
    history = []
    for step in range(1, n_steps + 1):
        src, dec_in, tgt = make_batch(batch_size, L, train_rng)
        logits = forward_fn(src, dec_in)
        loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), tgt.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, max_norm=5.0)
        optimizer.step()
        if step % 50 == 0 or step == 1:
            history.append((step, loss.item(), token_accuracy(logits, tgt)))
    return modules, forward_fn, history


def baseline_forward_factory():
    encoder = Encoder(VOCAB_SIZE, EMB_DIM, HIDDEN_DIM)
    decoder = BaselineDecoder(VOCAB_SIZE, EMB_DIM, HIDDEN_DIM)

    def forward(src, dec_in):
        _, h, c = encoder(src)
        return decoder(dec_in, h, c)

    return (encoder, decoder), forward


(baseline_encoder, baseline_decoder), baseline_forward, baseline_history = train_model(baseline_forward_factory)

## Adding Attention

The attention decoder is identical to the baseline except at every
step it also computes scaled dot-product attention (9a's exact formula,
batched) over *every* encoder position, using the current decoder hidden
state as the query, and feeds the resulting context vector into the
output projection alongside the hidden state itself:

$$\alpha = \text{softmax}\!\left(\frac{h_{\text{dec}} \, E^\top}{\sqrt{d}}\right), \qquad \text{context} = \alpha E, \qquad \text{logits} = W_{\text{out}}\, \tanh\!\big(W_c [\text{context}; h_{\text{dec}}]\big),$$

where $E$ is the encoder's full sequence of hidden states. Nothing about
the alignment is ever supervised — the model only ever sees a
cross-entropy loss on the output tokens, and the attention weights
themselves are a free side effect of what turns out to make that loss
small.

In [ ]:
class AttentionDecoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.cell = nn.LSTMCell(emb_dim, hidden_dim)
        self.combine = nn.Linear(hidden_dim * 2, hidden_dim)
        self.out = nn.Linear(hidden_dim, vocab_size)
        self.hidden_dim = hidden_dim

    def forward(self, decoder_in, h, c, encoder_outputs, mask=None):
        logits, attn_weights = [], []
        for t in range(decoder_in.shape[1]):
            h, c = self.cell(self.embed(decoder_in[:, t]), (h, c))
            scores = torch.bmm(encoder_outputs, h.unsqueeze(-1)).squeeze(-1) / self.hidden_dim ** 0.5
            if mask is not None:
                scores = scores.masked_fill(~mask, float("-inf"))
            weights = F.softmax(scores, dim=-1)
            context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)
            combined = torch.tanh(self.combine(torch.cat([context, h], dim=-1)))
            logits.append(self.out(combined))
            attn_weights.append(weights)
        return torch.stack(logits, dim=1), torch.stack(attn_weights, dim=1)


def attention_forward_factory():
    encoder = Encoder(VOCAB_SIZE, EMB_DIM, HIDDEN_DIM)
    decoder = AttentionDecoder(VOCAB_SIZE, EMB_DIM, HIDDEN_DIM)

    def forward(src, dec_in):
        encoder_outputs, h, c = encoder(src)
        logits, _ = decoder(dec_in, h, c, encoder_outputs)
        return logits

    return (encoder, decoder), forward


(attn_encoder, attn_decoder), attn_forward, attn_history = train_model(attention_forward_factory)

baseline_final_acc = baseline_history[-1][2]
attn_final_acc = attn_history[-1][2]
print(f"baseline (no attention) final token accuracy: {baseline_final_acc:.1%}")
print(f"attention model final token accuracy:          {attn_final_acc:.1%}")

In [ ]:
plt.figure()
plt.plot([s for s, l, a in baseline_history], [a for s, l, a in baseline_history], label="no attention")
plt.plot([s for s, l, a in attn_history], [a for s, l, a in attn_history], label="with attention")
plt.xlabel("training step")
plt.ylabel("token accuracy")
plt.title(f"Sequence reversal, L={L}")
plt.legend()
plt.tight_layout()
plt.show()

The attention model reaches substantially higher token accuracy on the
identical task, identical data and identical training budget — the only
architectural difference is whether the decoder can see every encoder
state or only the one compressed final vector, which is exactly the gap
9a's Jacobian-product argument predicted a long enough sequence would
expose.

## Attention Heatmaps

If the model actually learned the task's true structure rather than
some other trick, its attention weights should recover the task's real
alignment directly: decoder step $t$ should attend almost entirely to
encoder position $L-1-t$, since that is the one input token reversal
actually needs at that step.

In [ ]:
eval_rng = np.random.default_rng(123)
eval_src, eval_dec_in, eval_tgt = make_batch(1, L, eval_rng)
encoder_outputs, h0, c0 = attn_encoder(eval_src)
eval_logits, eval_attn = attn_decoder(eval_dec_in, h0, c0, encoder_outputs)
predicted = eval_logits.argmax(dim=-1)

print("source:    ", eval_src[0].tolist())
print("target:    ", eval_tgt[0].tolist())
print("predicted: ", predicted[0].tolist())
print(f"exact sequence match: {torch.equal(predicted[0], eval_tgt[0])}")

attn_matrix = eval_attn[0].detach().numpy()  # (decoder steps, encoder positions)
plt.figure(figsize=(6, 5))
plt.imshow(attn_matrix, cmap="viridis", aspect="auto")
plt.colorbar(label="attention weight")
plt.xlabel("encoder position")
plt.ylabel("decoder step")
plt.title("Learned attention alignment (reversal task)")
plt.plot(L - 1 - np.arange(L), np.arange(L), "r--", linewidth=1, label="expected alignment ($L-1-t$)")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
argmax_per_step = attn_matrix.argmax(axis=1)
expected_position = L - 1 - np.arange(L)
print("attended position per decoder step:", argmax_per_step.tolist())
print("expected position (L-1-t):         ", expected_position.tolist())
print(f"correlation(decoder step, attended position): {np.corrcoef(np.arange(L), argmax_per_step)[0, 1]:.2f}")

The attended position tracks the expected $L-1-t$ almost perfectly in
*trend* — a correlation near $-1$ between decoder step and attended
position, exactly the reversal the task needs — without landing on the
single exact expected index at every one of the 16 steps: a few
consecutive decoder steps sometimes share their peak attention across
adjacent encoder positions in a "staircase" rather than a perfect
one-to-one diagonal. That imprecision is consistent with the encoder
being a *recurrent* LSTM rather than a table of independent per-position
codes — 7a's own point about recurrence — so information about
neighbouring positions is already blended into each encoder state,
giving the model some slack in exactly which nearby index it peaks on
while still recovering the right answer. The model discovered this
alignment purely from a token-level cross-entropy loss, with no
alignment supervision of any kind ever provided.

### Masking

Real batches pad shorter sequences to a common length; those padding
positions carry no real content and must never receive attention
weight, or the model would be attending to noise. Comparing the same
attention computation with and without a mask over one padded example
makes the effect direct.

In [ ]:
real_len = 10
padded_src = torch.full((1, L), PAD, dtype=torch.long)
padded_src[0, :real_len] = torch.tensor(eval_rng.integers(0, 10, size=real_len))
mask = torch.zeros(1, L, dtype=torch.bool)
mask[0, :real_len] = True

encoder_outputs_p, h0_p, c0_p = attn_encoder(padded_src)
first_input = torch.full((1, 1), BOS, dtype=torch.long)

h_probe, c_probe = attn_decoder.cell(attn_decoder.embed(first_input[:, 0]), (h0_p, c0_p))
scores = torch.bmm(encoder_outputs_p, h_probe.unsqueeze(-1)).squeeze(-1) / HIDDEN_DIM ** 0.5
weights_unmasked = F.softmax(scores, dim=-1)[0]
weights_masked = F.softmax(scores.masked_fill(~mask, float("-inf")), dim=-1)[0]

print(f"real content: positions 0-{real_len - 1}; padding: positions {real_len}-{L - 1}")
print("unmasked weight on padding positions: ", weights_unmasked[real_len:].sum().item())
print("masked weight on padding positions:   ", weights_masked[real_len:].sum().item())

Without a mask, the softmax has no way to know which positions are
padding and spreads some genuine weight onto them; with the mask, their
score is forced to $-\infty$ before the softmax, so their weight is
exactly zero and the model's full attention budget goes to the real
content — the practical reason every production sequence model applies
a padding mask before every attention computation, not just an optional
refinement.

## Key Takeaways

- **An attention-augmented seq2seq model measurably beat a
  vanilla-bottleneck baseline** on the identical synthetic reversal task,
  identical data and identical training budget — the concrete, measured
  version of 9a's Jacobian-product argument about fixed-size encodings.
- **Learned attention weights recovered the task's true alignment**
  (decoder step $t$ attending to encoder position $L-1-t$) with no
  alignment supervision ever provided — only a token-level cross-entropy
  loss.
- **Masking padding positions is not optional**: without it, softmax
  spreads real attention weight onto content-free padding; with a mask
  forcing those scores to $-\infty$, their weight is exactly zero.